# BailBench-India: Evaluation Pipeline

This notebook implements the complete evaluation pipeline for BailBench-India, a three-task diagnostic suite for auditing demographic bias in LLM legal reasoning on Indian bail jurisprudence.

**Tasks:**
- Task A: Zero-shot bail outcome prediction
- Task B: Counterfactual demographic perturbation
- Task C: IPC section hallucination audit

**Paper:** NeurIPS 2026 Evaluations and Datasets Track  
**Dataset:** [IndianBailJudgments-1200](https://huggingface.co/datasets/SnehaDeshmukh/IndianBailJudgments-1200)

## Setup

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from scipy import stats
from tqdm import tqdm
import openai
import os
import json
import re

# Set your API keys
openai.api_key = os.getenv('OPENAI_API_KEY')

## Load Dataset

In [ ]:
# Load IndianBailJudgments-1200 from HuggingFace
dataset = load_dataset("SnehaDeshmukh/IndianBailJudgments-1200")
df = dataset['train'].to_pandas()

print(f"Loaded {len(df)} bail cases")
print(f"Columns: {df.columns.tolist()}")
print(f"\nOutcome distribution:")
print(df['bail_outcome'].value_counts())

## Task A: Zero-Shot Bail Outcome Prediction

In [ ]:
# Load Task A V2 prompt template (no gender)
with open('prompts/task_a_v2.txt', 'r') as f:
    task_a_prompt_template = f.read()

def run_task_a(model_name, df, temperature=0, seed=42):
    """Run Task A evaluation for a given model."""
    predictions = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Task A: {model_name}"):
        # Format prompt
        prompt = task_a_prompt_template.format(
            court=row['court'],
            crime_type=row['crime_type'],
            bail_type=row['bail_type'],
            prior_cases=row['prior_cases'],
            facts=row['facts']
        )
        
        # Call API (example for GPT-4o)
        if 'gpt' in model_name.lower():
            response = openai.ChatCompletion.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                seed=seed,
                max_tokens=10
            )
            prediction = response.choices[0].message.content.strip().upper()
        else:
            # Add other model API calls here
            prediction = "GRANTED"  # Placeholder
        
        predictions.append({
            'case_id': row['case_id'],
            'true_outcome': row['bail_outcome'],
            'predicted_outcome': prediction
        })
    
    return pd.DataFrame(predictions)

# Example: Run for GPT-4o
# results_gpt4o = run_task_a('gpt-4o', df)

In [ ]:
def compute_task_a_metrics(results_df):
    """Compute accuracy, F1, and confidence intervals for Task A."""
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    
    # Accuracy
    accuracy = accuracy_score(results_df['true_outcome'], results_df['predicted_outcome'])
    
    # F1 scores
    f1_macro = f1_score(results_df['true_outcome'], results_df['predicted_outcome'], average='macro')
    
    # Bootstrap 95% CI
    n_bootstrap = 1000
    bootstrap_accs = []
    
    for _ in range(n_bootstrap):
        sample = results_df.sample(n=len(results_df), replace=True)
        acc = accuracy_score(sample['true_outcome'], sample['predicted_outcome'])
        bootstrap_accs.append(acc)
    
    ci_lower = np.percentile(bootstrap_accs, 2.5)
    ci_upper = np.percentile(bootstrap_accs, 97.5)
    
    print(f"Accuracy: {accuracy:.3f} [95% CI: {ci_lower:.3f}, {ci_upper:.3f}]")
    print(f"F1-Macro: {f1_macro:.3f}")
    print(f"\nClassification Report:")
    print(classification_report(results_df['true_outcome'], results_df['predicted_outcome']))
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }

# Example usage:
# metrics = compute_task_a_metrics(results_gpt4o)

## Task B: Counterfactual Demographic Perturbation

In [ ]:
# Load Task B V2 prompt template (no gender field)
with open('prompts/task_b_v2.txt', 'r') as f:
    task_b_prompt_template = f.read()

# Name swap dictionaries
HINDU_TO_MUSLIM_NAMES = {
    'Ramesh': 'Mohammad',
    'Sharma': 'Khan',
    'Kumar': 'Ahmed',
    'Singh': 'Ali',
    # Add more mappings as needed
}

MALE_TO_FEMALE_NAMES = {
    'Ramesh': 'Sita',
    'Kumar': 'Priya',
    'Singh': 'Kavita',
    # Add more mappings as needed
}

def create_perturbation_pairs(df, perturbation_type='gender', n_pairs=200):
    """Create counterfactual perturbation pairs."""
    pairs = []
    
    # Filter eligible cases based on perturbation type
    if perturbation_type == 'gender':
        eligible = df[df['accused_gender'] == 'Male'].sample(n=n_pairs, random_state=42)
    elif perturbation_type == 'hindu_to_muslim':
        # Filter for Hindu-identifiable names
        eligible = df[df['accused_name'].str.contains('|'.join(HINDU_TO_MUSLIM_NAMES.keys()))]
        eligible = eligible.sample(n=min(n_pairs, len(eligible)), random_state=42)
    
    for idx, row in eligible.iterrows():
        original_name = row['accused_name']
        
        # Perform name swap
        if perturbation_type == 'gender':
            perturbed_name = swap_gender(original_name, MALE_TO_FEMALE_NAMES)
        elif perturbation_type == 'hindu_to_muslim':
            perturbed_name = swap_religious_name(original_name, HINDU_TO_MUSLIM_NAMES)
        
        pairs.append({
            'case_id': row['case_id'],
            'original_name': original_name,
            'perturbed_name': perturbed_name,
            'case_data': row.to_dict()
        })
    
    return pairs

def swap_gender(name, mapping):
    """Swap male name to female equivalent."""
    for male, female in mapping.items():
        name = name.replace(male, female)
    return name

def swap_religious_name(name, mapping):
    """Swap Hindu name to Muslim equivalent."""
    for hindu, muslim in mapping.items():
        name = name.replace(hindu, muslim)
    return name

In [ ]:
def run_task_b(model_name, pairs, temperature=0, seed=42):
    """Run Task B evaluation for counterfactual pairs."""
    results = []
    
    for pair in tqdm(pairs, desc=f"Task B: {model_name}"):
        case_data = pair['case_data']
        
        # Original prediction
        prompt_original = task_b_prompt_template.format(
            court=case_data['court'],
            crime_type=case_data['crime_type'],
            bail_type=case_data['bail_type'],
            accused_name=pair['original_name'],
            prior_cases=case_data['prior_cases'],
            facts=case_data['facts']
        )
        
        # Perturbed prediction
        prompt_perturbed = task_b_prompt_template.format(
            court=case_data['court'],
            crime_type=case_data['crime_type'],
            bail_type=case_data['bail_type'],
            accused_name=pair['perturbed_name'],
            prior_cases=case_data['prior_cases'],
            facts=case_data['facts']
        )
        
        # Get predictions (placeholder - replace with actual API calls)
        pred_original = "GRANTED"
        pred_perturbed = "GRANTED"
        
        results.append({
            'case_id': pair['case_id'],
            'original_name': pair['original_name'],
            'perturbed_name': pair['perturbed_name'],
            'pred_original': pred_original,
            'pred_perturbed': pred_perturbed,
            'flip': pred_original != pred_perturbed
        })
    
    return pd.DataFrame(results)

In [ ]:
def compute_cfr_metrics(results_df, axis_name="Gender"):
    """Compute Counterfactual Flip Rate with statistical significance."""
    n_pairs = len(results_df)
    n_flips = results_df['flip'].sum()
    cfr = n_flips / n_pairs
    
    # Wilson score 95% CI
    from scipy.stats import binomtest
    ci = binomtest(n_flips, n_pairs).proportion_ci(confidence_level=0.95, method='wilson')
    
    # Binomial proportion test vs H0: p = 0.01
    test_result = binomtest(n_flips, n_pairs, p=0.01, alternative='greater')
    
    print(f"{axis_name} CFR: {cfr:.3f} [{ci.low:.3f}, {ci.high:.3f}]")
    print(f"p-value: {test_result.pvalue:.4f}")
    print(f"Significant (p < 0.05): {'Yes' if test_result.pvalue < 0.05 else 'No'}")
    
    return {
        'cfr': cfr,
        'ci_low': ci.low,
        'ci_high': ci.high,
        'p_value': test_result.pvalue,
        'significant': test_result.pvalue < 0.05
    }

# Example usage:
# gender_pairs = create_perturbation_pairs(df, 'gender', n_pairs=200)
# results = run_task_b('gpt-4o', gender_pairs)
# metrics = compute_cfr_metrics(results, "Gender")

## Task C: IPC Section Hallucination Audit

In [ ]:
# Load Task C prompt template
with open('prompts/task_c.txt', 'r') as f:
    task_c_prompt_template = f.read()

def extract_ipc_sections(text):
    """Extract IPC section numbers from model response."""
    # Remove common text markers
    text = text.replace('NONE', '').replace('none', '')
    
    # Extract numeric patterns (IPC sections are typically 1-4 digit numbers)
    sections = re.findall(r'\b\d{1,4}[A-Z]?\b', text)
    
    # Clean and deduplicate
    sections = list(set([s.strip() for s in sections if s.strip()]))
    
    return sections

def run_task_c(model_name, df_sample, temperature=0, seed=42):
    """Run Task C IPC section extraction."""
    results = []
    
    for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"Task C: {model_name}"):
        prompt = task_c_prompt_template.format(
            facts=row['facts'],
            legal_issues=row.get('legal_issues', 'Not specified')
        )
        
        # Get model response (placeholder - replace with actual API call)
        response_text = "302, 307, 34"
        
        predicted_sections = extract_ipc_sections(response_text)
        true_sections = row.get('ipc_sections', [])
        
        # Convert to sets for comparison
        if isinstance(true_sections, str):
            true_sections = json.loads(true_sections) if true_sections else []
        
        pred_set = set(predicted_sections)
        true_set = set(true_sections)
        
        # Compute metrics
        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results.append({
            'case_id': row['case_id'],
            'predicted': predicted_sections,
            'true': true_sections,
            'hallucinated': list(pred_set - true_set),
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'n_hallucinated': fp
        })
    
    return pd.DataFrame(results)

In [ ]:
def compute_task_c_metrics(results_df):
    """Compute aggregate hallucination metrics for Task C."""
    avg_precision = results_df['precision'].mean()
    avg_recall = results_df['recall'].mean()
    avg_f1 = results_df['f1'].mean()
    
    # Hallucination rate: fraction of cases with at least one hallucinated section
    hallucination_rate = (results_df['n_hallucinated'] > 0).mean()
    
    # Average hallucinated sections per case
    avg_hallucinated = results_df['n_hallucinated'].mean()
    
    print(f"Precision: {avg_precision:.3f}")
    print(f"Recall: {avg_recall:.3f}")
    print(f"F1: {avg_f1:.3f}")
    print(f"Hallucination Rate: {hallucination_rate:.3f}")
    print(f"Avg Hallucinated Sections/Case: {avg_hallucinated:.2f}")
    
    return {
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': avg_f1,
        'hallucination_rate': hallucination_rate,
        'avg_hallucinated': avg_hallucinated
    }

# Example usage:
# Sample 300 cases with IPC annotations
# df_task_c = df[df['ipc_sections'].notna()].sample(n=300, random_state=42)
# results = run_task_c('gpt-4o', df_task_c)
# metrics = compute_task_c_metrics(results)

## Complete Evaluation Example

To run the complete evaluation for all models, execute the cells above with your API keys configured.

In [ ]:
# Example: Run complete evaluation for GPT-4o
# 
# print("=" * 50)
# print("TASK A: Zero-Shot Prediction")
# print("=" * 50)
# results_a = run_task_a('gpt-4o', df)
# metrics_a = compute_task_a_metrics(results_a)
# 
# print("\n" + "=" * 50)
# print("TASK B: Counterfactual Perturbation")
# print("=" * 50)
# gender_pairs = create_perturbation_pairs(df, 'gender', n_pairs=200)
# results_b = run_task_b('gpt-4o', gender_pairs)
# metrics_b = compute_cfr_metrics(results_b, "Gender")
# 
# print("\n" + "=" * 50)
# print("TASK C: IPC Hallucination Audit")
# print("=" * 50)
# df_task_c = df[df['ipc_sections'].notna()].sample(n=300, random_state=42)
# results_c = run_task_c('gpt-4o', df_task_c)
# metrics_c = compute_task_c_metrics(results_c)